# Data Wrangling: San Diego Neighborhood Opportunity Finder

This notebook starts the data wrangling process for the San Diego Neighborhood Opportunity Finder project. The goal is to build a clean tract-level dataset that can later be used to score areas based on development opportunity, livability, safety, mobility, and access to key neighborhood amenities.

In this notebook, I load the downloaded ACS data, inspect the structure, clean column names, check missing values, and identify the variables that are most useful for the project. This first version focuses on Census tract-level demographic, income, commute, household, and housing data. More location-based layers like crime, schools, parks, libraries, transit, and environmental risk will be added later.

## Datasets Used

**ACS 5-Year Data Profile**
- Source: U.S. Census Bureau American Community Survey
- Geography: Census tracts in San Diego County
- Main use: Demographics, income, commute patterns, housing, family structure, and vehicle access
- Notes: Census tracts are official Census areas. They are useful for analysis, but they do not always match local neighborhood names exactly.

## Notebook Goals

- Load the raw ACS data
- Review rows, columns, and data types
- Clean column names
- Identify useful variables for scoring
- Check missing values and duplicate rows
- Create a cleaned tract-level dataset for later analysis

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler, StandardScaler


pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

sns.set_theme(style='whitegrid', palette='Set2')

In [3]:
# path to the raw data folder
raw_path = Path(r'C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\raw')

dp02_raw = pd.read_csv(raw_path / 'dp02_2024.csv')
dp03_raw = pd.read_csv(raw_path / 'dp03_2024.csv')
dp04_raw = pd.read_csv(raw_path / 'dp04_2024.csv')
dp05_raw = pd.read_csv(raw_path / 'dp05_2024.csv')

# reading ACS metadata files
dp02_metadata = pd.read_csv(raw_path / 'dp02_2024_metadata.csv')
dp03_metadata = pd.read_csv(raw_path / 'dp03_2024_metadata.csv')
dp04_metadata = pd.read_csv(raw_path / 'dp04_2024_metadata.csv')
dp05_metadata = pd.read_csv(raw_path / 'dp05_2024_metadata.csv')

# checking the size
print('DP02 raw:', dp02_raw.shape)
print('DP03 raw:', dp03_raw.shape)
print('DP04 raw:', dp04_raw.shape)
print('DP05 raw:', dp05_raw.shape)

print('DP02 metadata:', dp02_metadata.shape)
print('DP03 metadata:', dp03_metadata.shape)
print('DP04 metadata:', dp04_metadata.shape)
print('DP05 metadata:', dp05_metadata.shape)

DP02 raw: (738, 619)
DP03 raw: (738, 551)
DP04 raw: (738, 575)
DP05 raw: (738, 435)
DP02 metadata: (618, 2)
DP03 metadata: (550, 2)
DP04 metadata: (574, 2)
DP05 metadata: (434, 2)


In [4]:
# previewing one table
dp03_raw.head()

,GEO_ID,NAME,DP03_0001E,DP03_0001M,DP03_0002E,DP03_0002M,DP03_0003E,DP03_0003M,DP03_0004E,DP03_0004M,DP03_0005E,DP03_0005M,DP03_0006E,DP03_0006M,DP03_0007E,DP03_0007M,DP03_0008E,DP03_0008M,DP03_0009E,DP03_0009M,DP03_0010E,DP03_0010M,DP03_0011E,DP03_0011M,DP03_0012E,DP03_0012M,DP03_0013E,DP03_0013M,DP03_0014E,DP03_0014M,DP03_0015E,DP03_0015M,DP03_0016E,DP03_0016M,DP03_0017E,DP03_0017M,DP03_0018E,DP03_0018M,DP03_0019E,DP03_0019M,DP03_0020E,DP03_0020M,DP03_0021E,DP03_0021M,DP03_0022E,DP03_0022M,DP03_0023E,DP03_0023M,DP03_0024E,DP03_0024M,...,DP03_0113PM,DP03_0114PE,DP03_0114PM,DP03_0115PE,DP03_0115PM,DP03_0116PE,DP03_0116PM,DP03_0117PE,DP03_0117PM,DP03_0118PE,DP03_0118PM,DP03_0119PE,DP03_0119PM,DP03_0120PE,DP03_0120PM,DP03_0121PE,DP03_0121PM,DP03_0122PE,DP03_0122PM,DP03_0123PE,DP03_0123PM,DP03_0124PE,DP03_0124PM,DP03_0125PE,DP03_0125PM,DP03_0126PE,DP03_0126PM,DP03_0127PE,DP03_0127PM,DP03_0128PE,DP03_0128PM,DP03_0129PE,DP03_0129PM,DP03_0130PE,DP03_0130PM,DP03_0131PE,DP03_0131PM,DP03_0132PE,DP03_0132PM,DP03_0133PE,DP03_0133PM,DP03_0134PE,DP03_0134PM,DP03_0135PE,DP03_0135PM,DP03_0136PE,DP03_0136PM,DP03_0137PE,DP03_0137PM,Unnamed: 550
0,Geography,Geographic Area Name,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Civilian labor force,Margin of Error!!EMPLOYMENT STATUS!!Civilian l...,Estimate!!EMPLOYMENT STATUS!!Civilian labor fo...,Margin of Error!!EMPLOYMENT STATUS!!Civilian l...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,Margin of Error!!EMPLOYMENT STATUS!!Females 16...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,Margin of Error!!EMPLOYMENT STATUS!!Females 16...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,Margin of Error!!EMPLOYMENT STATUS!!Females 16...,Estimate!!EMPLOYMENT STATUS!!Females 16 years ...,Margin of Error!!EMPLOYMENT STATUS!!Females 16...,Estimate!!EMPLOYMENT STATUS!!Own children of t...,Margin of Error!!EMPLOYMENT STATUS!!Own childr...,Estimate!!EMPLOYMENT STATUS!!Own children of t...,Margin of Error!!EMPLOYMENT STATUS!!Own childr...,Estimate!!EMPLOYMENT STATUS!!Own children of t...,Margin of Error!!EMPLOYMENT STATUS!!Own childr...,Estimate!!EMPLOYMENT STATUS!!Own children of t...,Margin of Error!!EMPLOYMENT STATUS!!Own childr...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,Estimate!!COMMUTING TO WORK!!Workers 16 years ...,Margin of Error!!COMMUTING TO WORK!!Workers 16...,...,Percent Margin of Error!!HEALTH INSURANCE COVE...,Percent!!HEALTH INSURANCE COVERAGE!!Civilian n...,Percent Margin of Error!!HEALTH INSURANCE COVE...,Percent!!HEALTH INSURANCE COVERAGE!!Civilian n...,Percent Margin of Error!!HEALTH INSURANCE COVE...,Percent!!HEALTH INSURANCE COVERAGE!!Civilian n...,Percent Margin of Error!!HEALTH INSURANCE COVE...,Percent!!HEALTH INSURANCE COVERAGE!!Civilian n...,Percent Margin of

In [5]:
# function to clean ACS tables from raw download
def clean_acs_table(df):
    clean_df = df.copy()
    
    # first row has readable labels
    readable_cols = clean_df.iloc[0].copy()
    readable_cols['GEO_ID'] = 'geo_id'
    readable_cols['NAME'] = 'tract_name'
    
    # applying readable column names
    clean_df.columns = readable_cols
    
    # removing the first row since it is now the header
    clean_df = clean_df.iloc[1:].copy()
    
    # cleaning column names
    clean_df.columns = (
        pd.Series(clean_df.columns)
        .astype(str)
        .str.lower()
        .str.replace('percent margin of error!!', 'percent_moe_', regex=False)
        .str.replace('margin of error!!', 'moe_', regex=False)
        .str.replace('estimate!!', '', regex=False)
        .str.replace('percent!!', 'pct_', regex=False)
        .str.replace(' ', '_', regex=False)
        .str.replace('!!', '_', regex=False)
        .str.replace(':', '', regex=False)
        .str.replace('-', '_', regex=False)
        .str.replace('/', '_', regex=False)
        .str.replace(',', '', regex=False)
        .str.replace('(', '', regex=False)
        .str.replace(')', '', regex=False)
        .str.replace('__', '_', regex=False)
        .str.strip('_')
        .tolist()
    )
    
    # removing duplicate column names
    clean_df = clean_df.loc[:, ~clean_df.columns.duplicated()].copy()
    
    # dropping margin of error columns
    clean_df = clean_df[
        [col for col in clean_df.columns 
         if not str(col).startswith('moe_') 
         and not str(col).startswith('percent_moe_')
         and not str(col).startswith('pct_moe_')]
    ].copy()
    
    # converting numeric columns
    for col in clean_df.columns:
        if col not in ['geo_id', 'tract_name']:
            clean_df[col] = (
                clean_df[col]
                .astype(str)
                .str.replace(',', '', regex=False)
                .str.replace('%', '', regex=False)
                .replace(['-', 'N', '(X)', 'nan'], np.nan)
            )
            clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')
    
    return clean_df

In [6]:
# cleaning each table
dp02 = clean_acs_table(dp02_raw)
dp03 = clean_acs_table(dp03_raw)
dp04 = clean_acs_table(dp04_raw)
dp05 = clean_acs_table(dp05_raw)

# checking shapes
print('DP02 cleaned:', dp02.shape)
print('DP03 cleaned:', dp03.shape)
print('DP04 cleaned:', dp04.shape)
print('DP05 cleaned:', dp05.shape)

DP02 cleaned: (737, 311)
DP03 cleaned: (737, 277)
DP04 cleaned: (737, 289)
DP05 cleaned: (737, 211)


In [7]:
dp02.columns.tolist()

['geo_id',
 'tract_name',
 'households_by_type_total_households',
 'households_by_type_total_households_married_couple_household',
 'households_by_type_total_households_married_couple_household_with_children_of_the_householder_under_18_years',
 'households_by_type_total_households_cohabiting_couple_household',
 'households_by_type_total_households_cohabiting_couple_household_with_children_of_the_householder_under_18_years',
 'households_by_type_total_households_male_householder_no_spouse_partner_present',
 'households_by_type_total_households_male_householder_no_spouse_partner_present_with_children_of_the_householder_under_18_years',
 'households_by_type_total_households_male_householder_no_spouse_partner_present_householder_living_alone',
 'households_by_type_total_households_male_householder_no_spouse_partner_present_householder_living_alone_65_years_and_over',
 'households_by_type_total_households_female_householder_no_spouse_partner_present',
 'households_by_type_total_households_f

In [8]:
dp03.columns.tolist()

['geo_id',
 'tract_name',
 'employment_status_population_16_years_and_over',
 'employment_status_population_16_years_and_over_in_labor_force',
 'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force',
 'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_employed',
 'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_unemployed',
 'employment_status_population_16_years_and_over_in_labor_force_armed_forces',
 'employment_status_population_16_years_and_over_not_in_labor_force',
 'employment_status_civilian_labor_force',
 'employment_status_civilian_labor_force_unemployment_rate',
 'employment_status_females_16_years_and_over',
 'employment_status_females_16_years_and_over_in_labor_force',
 'employment_status_females_16_years_and_over_in_labor_force_civilian_labor_force',
 'employment_status_females_16_years_and_over_in_labor_force_civilian_labor_force_employed',
 'employment_status_own_children_of

There's an overwhelming number of categories since they're so specific, so I'm going to try a different approach to cleaning them. 

<b>Market Support:</b> find total population, household count, median household income

<b>Family Demand:</b> find family households, households with children, children under 18


<b>Housing Need:</b> find renter %, vacancy %, rent burden

<b>Car Dependence:</b> find vehicles available, drive-alone commute %, mean commute time

<b>Equity Need:</b> find poverty %, unemployment %, no vehicle households

In [9]:
# function to search column names by keyword
def search_columns(df, keywords, max_results=20):
    for keyword in keywords:
        print(f'\n--- {keyword.upper()} ---')
        
        matches = [
            col for col in df.columns 
            if keyword in str(col).lower()]
        
        if len(matches) == 0:
            print('No matches found')
        else:
            for col in matches[:max_results]:
                print(col)

In [10]:
# searching each ACS table for useful project variables
keywords = ['income', 'poverty', 'commute', 'vehicle', 'children', 'family', 'rent', 'owner', 'household']

search_columns(dp02, keywords)
search_columns(dp03, keywords)
search_columns(dp04, keywords)
search_columns(dp05, keywords)


--- INCOME ---
No matches found

--- POVERTY ---
No matches found

--- COMMUTE ---
No matches found

--- VEHICLE ---
No matches found

--- CHILDREN ---
households_by_type_total_households_married_couple_household_with_children_of_the_householder_under_18_years
households_by_type_total_households_cohabiting_couple_household_with_children_of_the_householder_under_18_years
households_by_type_total_households_male_householder_no_spouse_partner_present_with_children_of_the_householder_under_18_years
households_by_type_total_households_female_householder_no_spouse_partner_present_with_children_of_the_householder_under_18_years
grandparents_number_of_grandparents_living_with_own_grandchildren_under_18_years
grandparents_number_of_grandparents_living_with_own_grandchildren_under_18_years_grandparents_responsible_for_grandchildren
grandparents_number_of_grandparents_living_with_own_grandchildren_under_18_years_years_responsible_for_grandchildren_less_than_1_year
grandparents_number_of_grandpar

In [11]:
# searching one table at a time
search_columns(dp03, ['median', 'income', 'poverty', 'commute', 'drove', 'public_transportation', 'worked_from_home'])


--- MEDIAN ---
income_and_benefits_in_2024_inflation_adjusted_dollars_total_households_median_household_income_dollars
income_and_benefits_in_2024_inflation_adjusted_dollars_families_median_family_income_dollars
income_and_benefits_in_2024_inflation_adjusted_dollars_nonfamily_households_median_nonfamily_income_dollars
income_and_benefits_in_2024_inflation_adjusted_dollars_median_earnings_for_workers_dollars
income_and_benefits_in_2024_inflation_adjusted_dollars_median_earnings_for_male_full_time_year_round_workers_dollars
income_and_benefits_in_2024_inflation_adjusted_dollars_median_earnings_for_female_full_time_year_round_workers_dollars
pct_income_and_benefits_in_2024_inflation_adjusted_dollars_total_households_median_household_income_dollars
pct_income_and_benefits_in_2024_inflation_adjusted_dollars_families_median_family_income_dollars
pct_income_and_benefits_in_2024_inflation_adjusted_dollars_nonfamily_households_median_nonfamily_income_dollars
pct_income_and_benefits_in_2024_inf

## Selecting Candidate ACS Features

The ACS tables include many more columns than I need for this project. Instead of keeping every field, I am selecting a smaller group of candidate variables that connect to the project goals.

They'll focus on population, household structure, income, commute behavior, housing mix, vehicle access, and family demand. This keep it manageable while still leaving enough information for EDA and scoring.

In [12]:
# create a function to search by keywork
def search_columns(df, keywords, max_results=30):
    for keyword in keywords:
        print(f'\n--- {keyword.upper()} ---')
        
        matches = [
            col for col in df.columns
            if keyword in str(col).lower()]
        
        if len(matches) == 0:
            print('No matches found')
        else:
            for col in matches[:max_results]:
                print(col)

In [13]:
# searching specific relevant keywords
dp02_keywords = ['household', 'family', 'children', 'education', 'school']
dp03_keywords = ['income', 'poverty', 'employed', 'unemployed', 'commute', 'drove', 'public_transportation', 'worked_from_home']
dp04_keywords = ['housing', 'occupied', 'vacant', 'owner', 'renter', 'vehicle', 'rent']
dp05_keywords = ['population', 'under_5', 'under_18', 'over_65', 'median_age', 'race', 'hispanic']

print('================ DP02: HOUSEHOLDS / FAMILIES ================')
search_columns(dp02, dp02_keywords)

print('\n================ DP03: INCOME / POVERTY / COMMUTE ================')
search_columns(dp03, dp03_keywords)

print('\n================ DP04: HOUSING / VEHICLES ================')
search_columns(dp04, dp04_keywords)

print('\n================ DP05: POPULATION / AGE ================')
search_columns(dp05, dp05_keywords)

================ DP02: HOUSEHOLDS / FAMILIES ================

--- HOUSEHOLD ---
households_by_type_total_households
households_by_type_total_households_married_couple_household
households_by_type_total_households_married_couple_household_with_children_of_the_householder_under_18_years
households_by_type_total_households_cohabiting_couple_household
households_by_type_total_households_cohabiting_couple_household_with_children_of_the_householder_under_18_years
households_by_type_total_households_male_householder_no_spouse_partner_present
households_by_type_total_households_male_householder_no_spouse_partner_present_with_children_of_the_householder_under_18_years
households_by_type_total_households_male_householder_no_spouse_partner_present_householder_living_alone
households_by_type_total_households_male_householder_no_spouse_partner_present_householder_living_alone_65_years_and_over
households_by_type_total_households_female_householder_no_spouse_partner_present
households_by_type_total

## Potential Variables

Thetables had a ton of columns, so I am selecting a smaller set of potential variables that connect directly to the project goals. They focus on population demand, family structure, income, commute behavior, housing mix, vehicle access, and age structure.

I am keeping variables that can help explain whether a census tract has market support, family demand, housing need, car dependence, or equity-related need. I am not using every field since they're too detailed. Margin of error columns are also excluded for now, but this is a limitation to note later.

In [14]:
# selected columns

dp02_cols = [
    'geo_id',
    'tract_name',
    'households_by_type_total_households',
    'households_by_type_total_households_households_with_one_or_more_people_under_18_years',
    'households_by_type_total_households_households_with_one_or_more_people_65_years_and_over',
    'households_by_type_total_households_average_household_size',
    'households_by_type_total_households_average_family_size',
    'school_enrollment_population_3_years_and_over_enrolled_in_school',
    'school_enrollment_population_3_years_and_over_enrolled_in_school_elementary_school_grades_1_8',
    'school_enrollment_population_3_years_and_over_enrolled_in_school_high_school_grades_9_12',
    "educational_attainment_population_25_years_and_over_bachelor's_degree_or_higher"]

dp03_cols = [
    'geo_id',
    'tract_name',
    'income_and_benefits_in_2024_inflation_adjusted_dollars_total_households_median_household_income_dollars',
    'income_and_benefits_in_2024_inflation_adjusted_dollars_total_households_mean_household_income_dollars',
    'pct_percentage_of_families_and_people_whose_income_in_the_past_12_months_is_below_the_poverty_level_all_people',
    'pct_percentage_of_families_and_people_whose_income_in_the_past_12_months_is_below_the_poverty_level_all_families',
    'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_employed',
    'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_unemployed',
    'pct_employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_unemployed',
    'commuting_to_work_workers_16_years_and_over_car_truck_or_van__drove_alone',
    'pct_commuting_to_work_workers_16_years_and_over_car_truck_or_van__drove_alone',
    'commuting_to_work_workers_16_years_and_over_public_transportation',
    'pct_commuting_to_work_workers_16_years_and_over_public_transportation',
    'commuting_to_work_workers_16_years_and_over_worked_from_home',
    'pct_commuting_to_work_workers_16_years_and_over_worked_from_home']

dp04_cols = [
    'geo_id',
    'tract_name',
    'housing_occupancy_total_housing_units',
    'housing_occupancy_total_housing_units_occupied_housing_units',
    'housing_occupancy_total_housing_units_vacant_housing_units',
    'pct_housing_occupancy_total_housing_units_vacant_housing_units',
    'housing_tenure_occupied_housing_units_owner_occupied',
    'housing_tenure_occupied_housing_units_renter_occupied',
    'pct_housing_tenure_occupied_housing_units_renter_occupied',
    'vehicles_available_occupied_housing_units',
    'vehicles_available_occupied_housing_units_no_vehicles_available',
    'pct_vehicles_available_occupied_housing_units_no_vehicles_available',
    'vehicles_available_occupied_housing_units_1_vehicle_available',
    'vehicles_available_occupied_housing_units_2_vehicles_available',
    'vehicles_available_occupied_housing_units_3_or_more_vehicles_available',
    'gross_rent_occupied_units_paying_rent_median_dollars',
    'gross_rent_as_a_percentage_of_household_income_grapi_occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed_30.0_to_34.9_percent',
    'gross_rent_as_a_percentage_of_household_income_grapi_occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed_35.0_percent_or_more']

dp05_cols = [
    'geo_id',
    'tract_name',
    'sex_and_age_total_population',
    'sex_and_age_total_population_under_5_years',
    'pct_sex_and_age_total_population_under_5_years',
    'sex_and_age_total_population_under_18_years',
    'pct_sex_and_age_total_population_under_18_years',
    'sex_and_age_total_population_65_years_and_over',
    'sex_and_age_total_population_median_age_years',
    'hispanic_or_latino_and_race_total_population_hispanic_or_latino_of_any_race',
    'pct_hispanic_or_latino_and_race_total_population_hispanic_or_latino_of_any_race']

In [15]:
# creating smaller selected datasets
dp02_selected = dp02[dp02_cols].copy()
dp03_selected = dp03[dp03_cols].copy()
dp04_selected = dp04[dp04_cols].copy()
dp05_selected = dp05[dp05_cols].copy()

# checking shapes
print('DP02 selected:', dp02_selected.shape)
print('DP03 selected:', dp03_selected.shape)
print('DP04 selected:', dp04_selected.shape)
print('DP05 selected:', dp05_selected.shape)

DP02 selected: (737, 11)
DP03 selected: (737, 15)
DP04 selected: (737, 18)
DP05 selected: (737, 11)


In [16]:
# renaming selected ACS columns into cleaner project-friendly names

dp02_rename = {
    'geo_id': 'geo_id',
    'tract_name': 'tract_name',
    'households_by_type_total_households': 'total_households',
    'households_by_type_total_households_households_with_one_or_more_people_under_18_years': 'households_with_children',
    'households_by_type_total_households_households_with_one_or_more_people_65_years_and_over': 'households_with_seniors',
    'households_by_type_total_households_average_household_size': 'avg_household_size',
    'households_by_type_total_households_average_family_size': 'avg_family_size',
    'school_enrollment_population_3_years_and_over_enrolled_in_school': 'school_enrolled_population',
    'school_enrollment_population_3_years_and_over_enrolled_in_school_elementary_school_grades_1_8': 'elementary_school_enrollment',
    'school_enrollment_population_3_years_and_over_enrolled_in_school_high_school_grades_9_12': 'high_school_enrollment',
    "educational_attainment_population_25_years_and_over_bachelor's_degree_or_higher": 'bachelors_or_higher'}

dp03_rename = {
    'geo_id': 'geo_id',
    'tract_name': 'tract_name',
    'income_and_benefits_in_2024_inflation_adjusted_dollars_total_households_median_household_income_dollars': 'median_household_income',
    'income_and_benefits_in_2024_inflation_adjusted_dollars_total_households_mean_household_income_dollars': 'mean_household_income',
    'pct_percentage_of_families_and_people_whose_income_in_the_past_12_months_is_below_the_poverty_level_all_people': 'poverty_rate',
    'pct_percentage_of_families_and_people_whose_income_in_the_past_12_months_is_below_the_poverty_level_all_families': 'family_poverty_rate',
    'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_employed': 'employed_population',
    'employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_unemployed': 'unemployed_population',
    'pct_employment_status_population_16_years_and_over_in_labor_force_civilian_labor_force_unemployed': 'unemployment_rate',
    'commuting_to_work_workers_16_years_and_over_car_truck_or_van__drove_alone': 'drove_alone_count',
    'pct_commuting_to_work_workers_16_years_and_over_car_truck_or_van__drove_alone': 'drove_alone_rate',
    'commuting_to_work_workers_16_years_and_over_public_transportation': 'public_transit_commute_count',
    'pct_commuting_to_work_workers_16_years_and_over_public_transportation': 'public_transit_commute_rate',
    'commuting_to_work_workers_16_years_and_over_worked_from_home': 'work_from_home_count',
    'pct_commuting_to_work_workers_16_years_and_over_worked_from_home': 'work_from_home_rate'}

dp04_rename = {
    'geo_id': 'geo_id',
    'tract_name': 'tract_name',
    'housing_occupancy_total_housing_units': 'total_housing_units',
    'housing_occupancy_total_housing_units_occupied_housing_units': 'occupied_housing_units',
    'housing_occupancy_total_housing_units_vacant_housing_units': 'vacant_housing_units',
    'pct_housing_occupancy_total_housing_units_vacant_housing_units': 'vacancy_rate',
    'housing_tenure_occupied_housing_units_owner_occupied': 'owner_occupied_units',
    'housing_tenure_occupied_housing_units_renter_occupied': 'renter_occupied_units',
    'pct_housing_tenure_occupied_housing_units_renter_occupied': 'renter_rate',
    'vehicles_available_occupied_housing_units': 'vehicle_households',
    'vehicles_available_occupied_housing_units_no_vehicles_available': 'no_vehicle_households',
    'pct_vehicles_available_occupied_housing_units_no_vehicles_available': 'no_vehicle_rate',
    'vehicles_available_occupied_housing_units_1_vehicle_available': 'one_vehicle_households',
    'vehicles_available_occupied_housing_units_2_vehicles_available': 'two_vehicle_households',
    'vehicles_available_occupied_housing_units_3_or_more_vehicles_available': 'three_plus_vehicle_households',
    'gross_rent_occupied_units_paying_rent_median_dollars': 'median_gross_rent',
    'gross_rent_as_a_percentage_of_household_income_grapi_occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed_30.0_to_34.9_percent': 'rent_burden_30_34_count',
    'gross_rent_as_a_percentage_of_household_income_grapi_occupied_units_paying_rent_excluding_units_where_grapi_cannot_be_computed_35.0_percent_or_more': 'rent_burden_35_plus_count'}

dp05_rename = {
    'geo_id': 'geo_id',
    'tract_name': 'tract_name',
    'sex_and_age_total_population': 'total_population',
    'sex_and_age_total_population_under_5_years': 'population_under_5',
    'pct_sex_and_age_total_population_under_5_years': 'population_under_5_rate',
    'sex_and_age_total_population_under_18_years': 'population_under_18',
    'pct_sex_and_age_total_population_under_18_years': 'population_under_18_rate',
    'sex_and_age_total_population_65_years_and_over': 'population_65_plus',
    'sex_and_age_total_population_median_age_years': 'median_age',
    'hispanic_or_latino_and_race_total_population_hispanic_or_latino_of_any_race': 'hispanic_latino_population',
    'pct_hispanic_or_latino_and_race_total_population_hispanic_or_latino_of_any_race': 'hispanic_latino_rate'}

# applying cleaner names
dp02_selected = dp02_selected.rename(columns=dp02_rename)
dp03_selected = dp03_selected.rename(columns=dp03_rename)
dp04_selected = dp04_selected.rename(columns=dp04_rename)
dp05_selected = dp05_selected.rename(columns=dp05_rename)

In [17]:
dp02_selected.describe().T

,count,mean,std,min,25%,50%,75%,max
total_households,737.00,"1,589.25",638.60,0.00,"1,162.00","1,542.00","1,946.00","7,257.00"
households_with_children,737.00,484.77,302.07,0.00,303.00,462.00,631.00,"4,062.00"
households_with_seniors,737.00,481.60,250.87,0.00,306.00,450.00,614.00,"1,730.00"
avg_household_size,731.00,2.77,0.60,1.39,2.41,2.78,3.15,6.36
avg_family_size,731.00,3.25,0.50,1.80,2.94,3.24,3.55,6.36
school_enrolled_population,737.00,"1,106.51",738.62,0.00,692.00,"1,023.00","1,356.00","10,533.00"
elementary_school_enrollment,737.00,411.23,283.21,0.00,234.00,369.00,546.00,"2,804.00"
high_school_enrollment,737.00,217.94,157.42,0.00,110.00,190.00,289.00,"1,208.00"
bachelors_or_higher,737.00,"1,336.01",818.59,0.00,664.00,"1,221.00","1,819.00","5,075.00"


In [18]:
dp03_selected.describe().T

,count,mean,std,min,25%,50%,75%,max
median_household_income,721.00,"113,782.69","41,994.25","34,148.00","81,085.00","109,750.00","138,866.00","247,222.00"
mean_household_income,729.00,"142,369.29","56,263.01","50,857.00","103,096.00","131,556.00","167,405.00","403,346.00"
poverty_rate,732.00,10.27,7.62,0.00,5.00,8.20,13.70,48.00
family_poverty_rate,731.00,7.08,6.50,0.00,2.30,5.10,10.10,32.60
employed_population,737.00,"2,152.19",844.59,0.00,"1,567.00","2,101.00","2,672.00","8,261.00"
unemployed_population,737.00,140.06,105.89,0.00,70.00,119.00,178.00,801.00
unemployment_rate,735.00,3.83,2.37,0.00,2.20,3.30,5.00,14.90
drove_alone_count,737.00,"1,462.02",685.22,0.00,"1,004.00","1,408.00","1,827.00","9,167.00"
drove_alone_rate,734.00,65.85,11.20,0.00,60.30,66.95,73.70,92.90
public_transit_commute_count,737.00,42.87,64.18,0.00,0.00,19.00,57.00,557.00


In [19]:
dp04_selected.describe().T

,count,mean,std,min,25%,50%,75%,max
total_housing_units,737.00,"1,696.93",675.37,0.00,"1,255.00","1,632.00","2,072.00","7,838.00"
occupied_housing_units,737.00,"1,589.25",638.60,0.00,"1,162.00","1,542.00","1,946.00","7,257.00"
vacant_housing_units,737.00,107.68,122.52,0.00,29.00,78.00,136.00,"1,059.00"
vacancy_rate,732.00,6.33,7.11,0.00,2.10,4.70,8.20,61.10
owner_occupied_units,737.00,867.22,503.86,0.00,498.00,809.00,"1,170.00","3,842.00"
renter_occupied_units,737.00,722.04,575.82,0.00,291.00,608.00,"1,057.00","7,181.00"
renter_rate,732.00,44.42,25.46,2.30,22.25,42.10,64.05,100.00
vehicle_households,737.00,"1,589.25",638.60,0.00,"1,162.00","1,542.00","1,946.00","7,257.00"
no_vehicle_households,737.00,87.26,104.09,0.00,21.00,55.00,119.00,"1,225.00"
no_vehicle_rate,732.00,5.51,6.64,0.00,1.60,3.70,7.30,100.00


In [20]:
dp05_selected.describe().T

,count,mean,std,min,25%,50%,75%,max
total_population,737.00,"4,462.38","2,120.31",0.00,"3,281.00","4,260.00","5,436.00","40,415.00"
population_under_5,737.00,249.14,227.39,0.00,117.00,214.00,323.00,"4,265.00"
population_under_5_rate,735.00,5.30,2.90,0.00,3.30,5.00,6.80,22.80
population_under_18,737.00,932.59,610.21,0.00,556.00,866.00,"1,212.00","8,443.00"
population_under_18_rate,735.00,20.05,7.39,0.00,16.50,20.90,24.50,49.70
population_65_plus,737.00,688.92,376.07,0.00,425.00,632.00,888.00,"2,567.00"
median_age,735.00,39.11,7.19,18.10,34.30,38.90,42.90,73.20
hispanic_latino_population,737.00,"1,545.54","1,273.20",0.00,644.00,"1,156.00","2,097.00","12,916.00"
hispanic_latino_rate,735.00,33.70,22.19,0.00,15.95,27.60,47.20,96.60


In [21]:
# merging selected ACS tables into one census dataframe
census_selected = (
    dp05_selected
    .merge(dp02_selected, on=['geo_id', 'tract_name'], how='left')
    .merge(dp03_selected, on=['geo_id', 'tract_name'], how='left')
    .merge(dp04_selected, on=['geo_id', 'tract_name'], how='left'))

# checking the merged dataset
print('Census selected shape:', census_selected.shape)
census_selected.head()

Census selected shape: (737, 49)


,geo_id,tract_name,total_population,population_under_5,population_under_5_rate,population_under_18,population_under_18_rate,population_65_plus,median_age,hispanic_latino_population,hispanic_latino_rate,total_households,households_with_children,households_with_seniors,avg_household_size,avg_family_size,school_enrolled_population,elementary_school_enrollment,high_school_enrollment,bachelors_or_higher,median_household_income,mean_household_income,poverty_rate,family_poverty_rate,employed_population,unemployed_population,unemployment_rate,drove_alone_count,drove_alone_rate,public_transit_commute_count,public_transit_commute_rate,work_from_home_count,work_from_home_rate,total_housing_units,occupied_housing_units,vacant_housing_units,vacancy_rate,owner_occupied_units,renter_occupied_units,renter_rate,vehicle_households,no_vehicle_households,no_vehicle_rate,one_vehicle_households,two_vehicle_households,three_plus_vehicle_households,median_gross_rent,rent_burden_30_34_count,rent_burden_35_plus_count
0,1400000US06073000100,Census Tract 1; San Diego County; California,2948,175,5.90,657,22.30,815,51.10,281,9.50,1178,318,543,2.50,2.89,716,224,229,1758,"231,667.00","331,240.00",2.20,2.10,1435,0,0.00,996,69.80,25,1.80,269,18.90,1293,1178,115,8.90,1067,111,9.40,1178,41,3.50,260,592,285,NaN,17,20
1,1400000US06073000201,Census Tract 2.01; San Diego County; California,2270,80,3.50,440,19.40,711,51.20,90,4.00,1180,208,577,1.92,3.03,459,212,111,1392,"124,722.00","224,327.00",6.70,6.50,1063,45,2.30,538,51.00,7,0.70,361,34.30,1261,1180,81,6.40,534,646,54.70,1180,148,12.50,557,353,122,"2,407.00",59,233
2,1400000US06073000202,Census Tract 2.02; San Diego County; California,3755,107,2.80,296,7.90,753,45.80,562,15.00,2240,185,557,1.64,2.44,343,99,39,2077,"120,091.00","144,236.00",2.90,0.90,2418,174,5.00,1424,58.00,117,4.80,816,33.20,2476,2240,236,9.50,1003,1237,55.20,2240,99,4.40,1210,703,228,"1,992.00",134,375
3,1400000US06073000301,Census Tract 3.01; San Diego County; California,2311,93,4.00,187,8.10,357,37.30,516,22.30,1302,131,250,1.77,2.55,244,32,27,1248,"87,813.00","121,130.00",15.40,18.10,1435,88,4.10,839,56.90,87,5.90,457,31.00,1475,1302,173,11.70,345,957,73.50,1302,81,6.20,683,436,102,"1,945.00",69,316
4,1400000US06073000302,Census Tract 3.02; San Diego County; California,2873,0,0.00,73,2.50,731,45.10,450,15.70,1818,63,535,1.48,1.99,131,38,0,1644,"89,573.00","125,762.00",9.40,0.00,1810,63,2.30,895,49.10,77,4.20,506,27.80,2197,1818,379,17.30,501,1317,72.40,1818,122,6.70,1290,342,64,"2,412.00",80,664


In [23]:
# checking missing values in the merged census dataset
missing_by_column = census_selected.isna().sum()
missing_by_column = missing_by_column[missing_by_column > 0].sort_values(ascending=False)

total_missing = missing_by_column.sum()
missing_pct = (total_missing / census_selected.size) * 100

print(f'Total missing values: {total_missing}')
print(f'Percent of dataset missing: {missing_pct:.2f}%')
print(f'Columns with missing values: {len(missing_by_column)}')

missing_by_column

Total missing values: 168
Percent of dataset missing: 0.47%
Columns with missing values: 18


median_gross_rent              87
median_household_income        16
mean_household_income           8
avg_household_size              6
family_poverty_rate             6
avg_family_size                 6
vacancy_rate                    5
renter_rate                     5
no_vehicle_rate                 5
poverty_rate                    5
drove_alone_rate                3
work_from_home_rate             3
public_transit_commute_rate     3
population_under_5_rate         2
population_under_18_rate        2
median_age                      2
hispanic_latino_rate            2
unemployment_rate               2
dtype: int64

In [24]:
# checking how many rows have at least one missing value
rows_with_missing = census_selected.isna().any(axis=1).sum()

print(f'Rows with at least one missing value: {rows_with_missing}')

Rows with at least one missing value: 87


In [25]:
# checking duplicate census tracts
duplicate_tracts = census_selected.duplicated(subset='geo_id').sum()

print(f'Duplicate geo_id rows: {duplicate_tracts}')

Duplicate geo_id rows: 0


In [27]:
# looking at rows with missing values
rows_missing = census_selected[census_selected.isna().any(axis=1)]

rows_missing[['geo_id', 'tract_name'] + missing_by_column.index.tolist()].head(20).T

,0,90,115,124,125,127,131,134,158,159,160,162,164,165,167,168,169,170,172,173
geo_id,1400000US06073000100,1400000US06073003800,1400000US06073005500,1400000US06073006200,1400000US06073006300,1400000US06073006600,1400000US06073007002,1400000US06073007302,1400000US06073008202,1400000US06073008301,1400000US06073008303,1400000US06073008306,1400000US06073008310,1400000US06073008311,1400000US06073008313,1400000US06073008324,1400000US06073008327,1400000US06073008328,1400000US06073008331,1400000US06073008336
tract_name,Census Tract 1; San Diego County; California,Census Tract 38; San Diego County; California,Census Tract 55; San Diego County; California,Census Tract 62; San Diego County; California,Census Tract 63; San Diego County; California,Census Tract 66; San Diego County; California,Census Tract 70.02; San Diego County; California,Census Tract 73.02; San Diego County; California,Census Tract 82.02; San Diego County; California,Census Tract 83.01; San Diego County; California,Census Tract 83.03; San Diego County; California,Census Tract 83.06; San Diego County; California,Census Tract 83.10; San Diego County; California,Census Tract 83.11; San Diego County; California,Census Tract 83.13; San Diego County; California,Census Tract 83.24; San Diego County; California,Census Tract 83.27; San Diego County; California,Census Tract 83.28; San Diego County; California,Census Tract 83.31; San Diego County; California,Census Tract 83.36; San Diego County; California
median_gross_rent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
median_household_income,"231,667.00",NaN,NaN,NaN,NaN,"110,651.00","184,167.00","183,529.00","111,836.00","216,823.00","230,511.00","204,779.00","172,399.00",NaN,"221,536.00","197,535.00","192,234.00",NaN,"230,956.00","190,938.00"
mean_household_income,"331,240.00",NaN,NaN,NaN,NaN,"116,733.00","228,668.00","260,650.00","187,844.00","251,841.00","264,998.00","224,645.00","248,661.00","398,422.00","278,684.00","243,599.00","215,373.00","356,214.00","254,026.00","203,256.00"
avg_household_size,2.50,NaN,6.36,NaN,NaN,4.26,2.41,2.65,1.68,2.31,2.28,2.48,2.65,2.50,2.71,2.39,2.43,3.03,2.80,2.70
family_poverty_rate,2.10,NaN,0.00,NaN,NaN,7.90,1.30,2.10,0.00,0.00,6.10,7.10,2.80,0.90,5.20,2.30,2.20,4.70,1.70,1.90
avg_family_size,2.89,NaN,6.36,NaN,NaN,4.52,2.96,3.00,2.36,2.74,2.77,2.94,3.03,2.83,3.00,2.90,2.94,3.24,3.07,2.94
vacancy_rate,8.90,44.40,0.00,NaN,NaN,26.60,3.20,4.60,38.30,0.00,19.70,2.10,4.20,9.00,10.40,6.90,4.10,13.60,3.50,0.00
renter_rate,9.40,100.00,100.00,NaN,NaN,100.00,11.80,20.70,63.70,16.90,20.10,8.00,7.50,4.70,13.80,20.40,26.40,10.40,9.90,5.80


In [28]:
# tracts you want to inspect
tracts_to_check = [
    'Census Tract 1; San Diego County; California',
    'Census Tract 38; San Diego County; California',
    'Census Tract 55; San Diego County; California',
    'Census Tract 62; San Diego County; California',
    'Census Tract 63; San Diego County; California',
    'Census Tract 66; San Diego County; California',
    'Census Tract 70.02; San Diego County; California',
    'Census Tract 73.02; San Diego County; California',
    'Census Tract 82.02; San Diego County; California',
    'Census Tract 83.01; San Diego County; California',
    'Census Tract 83.03; San Diego County; California',
    'Census Tract 83.06; San Diego County; California',
    'Census Tract 83.10; San Diego County; California',
    'Census Tract 83.11; San Diego County; California',
    'Census Tract 83.13; San Diego County; California',
    'Census Tract 83.24; San Diego County; California',
    'Census Tract 83.27; San Diego County; California',
    'Census Tract 83.28; San Diego County; California',
    'Census Tract 83.31; San Diego County; California',
    'Census Tract 83.36; San Diego County; California'
]

# checking population and housing context
tract_check = census_selected[
    census_selected['tract_name'].isin(tracts_to_check)
][[
    'geo_id',
    'tract_name',
    'total_population',
    'total_households',
    'total_housing_units',
    'occupied_housing_units',
    'median_household_income',
    'median_gross_rent']].copy()

tract_check

,geo_id,tract_name,total_population,total_households,total_housing_units,occupied_housing_units,median_household_income,median_gross_rent
0,1400000US06073000100,Census Tract 1; San Diego County; California,2948,1178,1293,1178,"231,667.00",NaN
90,1400000US06073003800,Census Tract 38; San Diego County; California,4573,15,27,15,NaN,NaN
115,1400000US06073005500,Census Tract 55; San Diego County; California,290,14,14,14,NaN,NaN
124,1400000US06073006200,Census Tract 62; San Diego County; California,28,0,0,0,NaN,NaN
125,1400000US06073006300,Census Tract 63; San Diego County; California,2038,0,0,0,NaN,NaN
127,1400000US06073006600,Census Tract 66; San Diego County; California,2032,477,650,477,"110,651.00",NaN
131,1400000US06073007002,Census Tract 70.02; San Diego County; California,3025,1256,1298,1256,"184,167.00",NaN
134,1400000US06073007302,Census Tract 73.02; San Diego County; California,2610,984,1031,984,"183,529.00",NaN
158,1400000US06073008202,Census Tract 82.02; San Diego County; California,1203,659,1068,659,"111,836.00",NaN
159,1400000US06073008301,Census Tract 83.01; San Diego County; California,3190,1347,1347,1347,"216,823.00",NaN


In [30]:
# flagging likely non-residential tracts
tract_check['tract_type_flag'] = np.select(
    [
        (tract_check['total_population'] < 100) & (tract_check['total_housing_units'] < 50),
        (tract_check['total_population'] >= 100) & (tract_check['total_housing_units'] < 50),
        (tract_check['total_housing_units'] >= 50)
    ],
    [
        'likely_non_residential_or_commercial',
        'likely_institutional_or_group_quarters',
        'residential_or_mixed'
    ],
    default='needs_review')

tract_check.sort_values(['tract_type_flag', 'total_population'])

,geo_id,tract_name,total_population,total_households,total_housing_units,occupied_housing_units,median_household_income,median_gross_rent,tract_type_flag
115,1400000US06073005500,Census Tract 55; San Diego County; California,290,14,14,14,NaN,NaN,likely_institutional_or_group_quarters
125,1400000US06073006300,Census Tract 63; San Diego County; California,2038,0,0,0,NaN,NaN,likely_institutional_or_group_quarters
90,1400000US06073003800,Census Tract 38; San Diego County; California,4573,15,27,15,NaN,NaN,likely_institutional_or_group_quarters
124,1400000US06073006200,Census Tract 62; San Diego County; California,28,0,0,0,NaN,NaN,likely_non_residential_or_commercial
158,1400000US06073008202,Census Tract 82.02; San Diego County; California,1203,659,1068,659,"111,836.00",NaN,residential_or_mixed
127,1400000US06073006600,Census Tract 66; San Diego County; California,2032,477,650,477,"110,651.00",NaN,residential_or_mixed
173,1400000US06073008336,Census Tract 83.36; San Diego County; California,2047,757,757,757,"190,938.00",NaN,residential_or_mixed
167,1400000US06073008313,Census Tract 83.13; San Diego County; California,2195,809,903,809,"221,536.00",NaN,residential_or_mixed
134,1400000US06073007302,Census Tract 73.02; San Diego County; California,2610,984,1031,984,"183,529.00",NaN,residential_or_mixed
172,1400000US06073008331,Census Tract 83.31; San Diego County; California,2648,945,979,945,"230,956.00",NaN,residential_or_mixed


In [33]:
# adding a simple tract type flag to the full census dataset
census_selected['tract_type_flag'] = np.select(
    [
        (census_selected['total_population'] < 100) & (census_selected['total_housing_units'] < 50),
        (census_selected['total_population'] >= 100) & (census_selected['total_housing_units'] < 50),
        (census_selected['total_housing_units'] >= 50)
    ],
    [
        'likely_non_residential',
        'likely_institutional_or_group_quarters',
        'residential_or_mixed'
    ],
    default='needs_review')

# checking the flag counts
census_selected['tract_type_flag'].value_counts()

tract_type_flag
residential_or_mixed                      727
likely_institutional_or_group_quarters      7
likely_non_residential                      3
Name: count, dtype: int64

In [34]:
# saving cleaned tract dataset
processed_path = Path(r'C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\processed')

census_selected.to_csv(processed_path / 'acs_tracts_selected_2024.csv', index=False)

## Wrangling Summary

I started with four Census data tables: DP02, DP03, DP04, and DP05. They covered household structure, income, commute behavior, housing, vehicle access, population, and age.

The raw files had a lot of specific, detailed columns, so I selected a small group of variables that connect to the project goals and renamed them into clearer columns. They'll be used later to help engineer scores for market strength, family demand, housing need, car dependence, and equity need.

Next, I merged the tables into one tract-level df. I also checked missing values, duplicate tract IDs, and flagged "likely non-residential" or "institutional tracts" based on population and housing counts. I'll use this cleaned file as the base dataset before adding safety, schools, parks, libraries, transit, and environmental layers.